# Part I : Data Collection - REE API

The objective of this session is to understand how data can be retrieved from specific sources as REE API (https://www.ree.es/en/apidatos).

To do so, we will use `requests` library and `json` libraries from Python. These libraries are considered as wrappers. 

Later on, we will use specific libraries for plotting as `matplotlib` and `datetime`. 

In [ ]:
# import requests library
import requests
import json
# import plotting library
import matplotlib
import matplotlib.pyplot as plt 
from pandas.plotting import register_matplotlib_converters
register_matplotlib_converters()
import datetime
# import pandas library
import pandas as pd

When we work with APIs, we have to write the **endpoint** we want to access. 
We need to write also the **folder/section** we want to access. We can have the same endpoint and different sections from where to get information. 

Most of the times, we need to add some **headers** to our requests that are required to get the information.

#### REE API requests
**Requests overview**

`GET /{lang}/datos/{category}/{widget}?[query]`

* `lang`:  Defines the response language.
* `category`: Defines the general category.
* `widget`: Defines the particular widget to be retrieved.
* `query`: Set of parameters used to filter the requested data.

**Query definition**
* `start_date`: Defines the starting date in ISO 8601 format: (`YYYY-MM-DDTHH:MM`)
* `end_date`: Defines the ending date in ISO 8601 format.
* `time_trunc`: Defines the time aggregation of the requested data. Valid values are: hour, day, month, year.
* `geo_trunc`: (Optional) Defines the geographical scope of the requested data.
* `geo_limit`: (Optional) Defines the electrical system of the requested data.
* `geo_ids`: (Optional) Defines the ID of the previously defined autonomous community/electrical system.

In [ ]:
endpoint = 'https://apidatos.ree.es'
get_archives = '/es/datos/mercados/precios-mercados-tiempo-real'
headers = {'Accept': 'application/json',
           'Content-Type': 'application/json',
           'Host': 'apidatos.ree.es'}
params = {'start_date': '2022-01-25T00:00', 'end_date': '2022-02-20T23:00', 'time_trunc':'hour'}

Then, we can write the request according to the requests library and get callable object.

In [ ]:
request = requests.get(endpoint+get_archives, headers=headers, params=params)

After that, we can check if the request is correct or not according to the response we get from the API. 

In [ ]:
request

In this case, the response we get is **200**, meaning that is has been accepted and the connection is established. 
There are different codes that tell us which is our connection status between us and the API. 

In [ ]:
status = request.status_code

In [ ]:
#Check the status code
if status < 200:
    print('informational')
    # If the status code is 200, treat the information.
elif status >= 200 and status < 300:
    print('Connection is established')
    #okBehavior(response) # runs the function to get list of archives
elif status >= 300 and status < 400:
    print('redirection')
elif status >= 400 and status < 500:
    print('client error')
else:
    print('server error')

The data can be accessed by means of a json file. By doing this, we can see all the IDs available for us to obtain data. 

In [ ]:
# accessing data by means of a json object
json = request.json()

In [ ]:
json

In [ ]:
PVPC = json['included'][0]
values = PVPC['attributes']['values']
values

In [ ]:
prices = []
hours = []

In [ ]:
for time_period in values:
    hour = time_period['datetime'][0:16]
    prices.append(time_period['value'])
    hours.append(hour)
    print(f"Price at {hour} is {time_period['value']} €/MWh")

In [ ]:
prices_df = pd.DataFrame({'Datetime': hours, 'Price': prices})

In [ ]:
prices_df.head()

In [ ]:
prices_df["Datetime"] = pd.to_datetime(prices_df["Datetime"], format='%Y%m%d %H:%M')
prices_df.dtypes

In [ ]:
prices_df = prices_df.set_index("Datetime")
prices_df.head()

### Final PVPC price

In [ ]:
matplotlib.rc('xtick', labelsize=15)     
matplotlib.rc('ytick', labelsize=15)

plt.style.use('ggplot')
plt.figure(figsize=(20,10))
plt.plot(prices_df,'r',linewidth=1.5)
plt.title(f"PVPC prices", fontsize=20)
plt.xlabel('Hour',fontsize=20 )
plt.ylabel('Price [€/MWh]',fontsize=20)
plt.show()

## Day-Ahead Market Clearing Price 

In [ ]:
#date_today = datetime.date.today()                # use todays prices
#date = date_today + datetime.timedelta(days=2)    # open to use tomorrows prices
#dateend = datetime.date.today()            # use to specify a date
#dateend = date                              # to use only one day

### Tomorrows prices can only be requested after 20:15 the day before. ###

#startdate = str(date_today) + "T00:00:00.00" # can specify more if desired
#enddate = str(dateend) + "T23:50:00.00"
indicator = str(600)    # market clearing (DAM) price. Imbalance is 687

# You can get it by putting the mouse over the indicator name on the webpage. 
# webpage: https://www.esios.ree.es/es/analisis/600
website = 'https://api.esios.ree.es/indicators/'+indicator+'?start_date='+'2019-10-12T00:00:00.00'+'&end_date='+'2019-10-13T00:00:00.00'

     
     
#print('Checking dates: ' , startdate , "to" , enddate) # printing the date 

URL = website # host website
GET = '/archives_json' # API link
HEADERS = {
            'Accept': "application/json; application/vnd.esios-api-v1+json",
            'Host': 'api.esios.ree.es',
            'Authorization': "Token token=\"a6f2f926dea90ade64acc97b3b4fff73af5bdc5d7bce554a1adfa16d554ede81",#"a6f2f926dea90ade64acc97b3b4fff73af5bdc5d7bce554a1adfa16d554ede81"',
            'Content-Type': 'application/json'}
#PARAMS = {'date':date}

# Runs the request to get the total URL with access token
response = requests.get(url = URL+GET, headers = HEADERS)#, params = PARAMS)

# Read the status code
status = response.status_code  

In [ ]:
status

In [ ]:
# Diving into the data:
marketlist = []
for stuff in response.json()["indicator"]['values']:
    #print(stuff) ## show more stuff that can be gathered
    if stuff['geo_id'] == 3: ## choose Espana. 1 = Portugal, 2 = Francia
    #print(stuff['value'])
        marketlist.append(stuff['value'])

        
print('\n')
print(f"List of market prices for day 2019-10-12 in €/MWh:")        
print(marketlist)

plt.style.use('ggplot')
plt.figure(figsize=(20,10))
plt.plot(marketlist,'r',linewidth=3, color='cornflowerblue')
plt.title(f"Day-Ahead Market prices for 2019-10-12")
plt.xlim(0,24)
plt.ylim(0,50)
plt.xlabel('Hour')
plt.ylabel('Price [€/MWh]')
plt.show()

In [ ]:
date_today = datetime.date.today()                # use todays prices
date = date_today + datetime.timedelta(days=2)    # open to use tomorrows prices
#dateend = datetime.date.today()            # use to specify a date
dateend = date                              # to use only one day

### Tomorrows prices can only be requested after 20:15 the day before. ###

startdate = str(date_today) + "T00:00:00.00" # can specify more if desired
enddate = str(dateend) + "T23:50:00.00"
indicator = str(600)    # market clearing (DAM) price. Imbalance is 687

# You can get it by putting the mouse over the indicator name on the webpage. 
# webpage: https://www.esios.ree.es/es/analisis/600
website = 'https://api.esios.ree.es/indicators/'+indicator+'?start_date='+startdate+'&end_date='+enddate
     
     
print('Checking dates: ' , startdate , "to" , enddate) # printing the date 

URL = website # host website
GET = '/archives_json' # API link
HEADERS = {
            'Accept': "application/json; application/vnd.esios-api-v1+json",
            'Host': 'api.esios.ree.es',
            'Authorization': "Token token=\"a6f2f926dea90ade64acc97b3b4fff73af5bdc5d7bce554a1adfa16d554ede81",#"a6f2f926dea90ade64acc97b3b4fff73af5bdc5d7bce554a1adfa16d554ede81"',
            'Content-Type': 'application/json'}
#PARAMS = {'date':date}

# Runs the request to get the total URL with access token
response = requests.get(url = URL+GET, headers = HEADERS)#, params = PARAMS)

# Read the status code
status = response.status_code  



In [ ]:
# Diving into the data:
marketlist = []
for stuff in response.json()["indicator"]['values']:
    #print(stuff) ## show more stuff that can be gathered
    if stuff['geo_id'] == 3: ## choose Espana. 1 = Portugal, 2 = Francia
    #print(stuff['value'])
        marketlist.append(stuff['value'])

        
print('\n')
print(f"List of market prices for day {date} in €/MWh:")        
print(marketlist)

plt.style.use('ggplot')
plt.figure(figsize=(20,10))
plt.plot(marketlist,'r',linewidth=3, color='cornflowerblue')
plt.title(f"Day-Ahead Market prices for {date}")
plt.xlabel('Hour')
plt.ylabel('Price [€/MWh]')
plt.show()

In [ ]:
len(marketlist)